In [1]:
# Dependencies
import pandas as pd
import numpy as np

from pathlib import Path

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator

from IPython.display import display

## Load Data

In [2]:
DATA_DIR = Path("../dataset/dataset_curated")

datasets = {
    "EGFR": pd.read_csv(DATA_DIR / "EGFR_curated.csv"),
    "BRAF": pd.read_csv(DATA_DIR / "BRAF_curated.csv")
}

for target, df in datasets.items():
    print(f"\n{'='*60}")
    print(target)
    print(f"{'='*60}")
    print(f"Shape        : {df.shape}")
    print(f"Missing      : {df.isna().sum().sum()}")
    print(f"Unique SMILES: {df['canonical_smiles'].nunique():,}")
    print(f"pIC50 range  : {df['pIC50'].min():.4f} - {df['pIC50'].max():.4f}")
    print("\nColumns:")
    print(df.columns.tolist())

    display(df.head())


EGFR
Shape        : (10304, 5)
Missing      : 0
Unique SMILES: 10,304
pIC50 range  : 3.0000 - 11.5229

Columns:
['canonical_smiles', 'molecule_chembl_id', 'pIC50', 'mean_pIC50', 'n_measurements']


,canonical_smiles,molecule_chembl_id,pIC50,mean_pIC50,n_measurements
0,Brc1cc2c(NCc3ccccc3)ncnc2s1,CHEMBL3416599,6.617983,6.617983,1
1,Brc1cc2c(NCc3ccccn3)ncnc2s1,CHEMBL3416616,5.102153,5.102153,1
2,Brc1cc2c(NCc3cccs3)ncnc2s1,CHEMBL3416619,5.862013,5.862013,1
3,Brc1cc2c(NCc3ccncc3)ncnc2s1,CHEMBL3416614,5.410833,5.410833,1
4,Brc1cc2c(Nc3ccccc3)ncnc2s1,CHEMBL3416621,7.096910,7.096910,1



BRAF
Shape        : (5353, 5)
Missing      : 0
Unique SMILES: 5,353
pIC50 range  : 3.0857 - 10.6990

Columns:
['canonical_smiles', 'molecule_chembl_id', 'pIC50', 'mean_pIC50', 'n_measurements']


,canonical_smiles,molecule_chembl_id,pIC50,mean_pIC50,n_measurements
0,C#CC(C)(C)c1cc(N=C(C)C)cc(C(=O)Nc2ccc(C)c(C(=O...,CHEMBL475818,7.119186,7.119186,1
1,C#CC(C)(C)c1cccc(C(=O)Nc2ccc(C)c(Nc3ccc4ncn(C)...,CHEMBL4203441,7.646215,7.646215,2
2,C#Cc1c(Nc2c(F)ccc(NS(=O)(=O)CCC)c2Cl)ccc2ncn(C...,CHEMBL5896210,8.033858,8.033858,2
3,C#Cc1cccc(N2C(=O)c3ccccc3C2(O)c2ccc3c(c2)NC(=O...,CHEMBL3983221,7.000000,7.000000,1
4,C#Cc1cccc(NC(=O)N[C@@H]2[C@H]3Oc4ccc(Oc5ccnc6c...,CHEMBL5851239,5.721246,5.721246,1


## Molecular Representation

Penelitian ini menggunakan representasi molekuler hybrid yang terdiri atas
**Morgan Fingerprint ECFP4 (2048 bit)** dan **10 molecular descriptors**.
Representasi ini dipilih untuk mempertahankan informasi struktur lokal sekaligus
karakteristik fisikokimia global dari setiap senyawa.

### 1. Morgan Fingerprint — ECFP4

ECFP4 dibentuk menggunakan Morgan fingerprint dengan:

- Radius = 2
- Ukuran fingerprint = 2048 bit

ECFP4 merepresentasikan lingkungan atom dan substruktur lokal dalam molekul
sebagai vektor biner. Representasi ini dapat menangkap keberadaan dan pola
substruktur, termasuk ring system, functional groups, serta lingkungan atom
yang relevan terhadap hubungan structure–activity.

Alshehade et al. menggunakan ECFP4 sebagai salah satu representasi utama
dalam benchmarking QSAR dan menunjukkan bahwa fingerprint dapat menangkap
informasi struktur dengan resolusi lebih tinggi dibandingkan descriptor
fisikokimia yang bersifat global. Mereka juga membentuk representasi hybrid
dengan menggabungkan RDKit 2D descriptors dan ECFP4 untuk memanfaatkan
informasi fisikokimia kontinu dan pola struktur molekuler secara bersamaan.
:chatgpt-content-reference{index="2"}

ECFP4 juga dipertahankan sebagai representasi struktur pada penelitian ini
karena digunakan kembali pada analisis **Tanimoto similarity/distance** untuk
applicability domain dan identifikasi activity cliff. Dengan demikian,
representasi struktur yang digunakan untuk pemodelan tetap konsisten dengan
representasi yang digunakan untuk analisis ruang kimia.

### 2. Molecular Descriptors

Selain fingerprint, digunakan 10 descriptor molekuler yang merepresentasikan
karakteristik fisikokimia dan struktural global:

| No. | Descriptor | Informasi yang direpresentasikan |
|---|---|---|
| 1 | Molecular Weight (MW) | Ukuran/massa molekul |
| 2 | Heavy Atom Count | Jumlah atom non-hidrogen |
| 3 | Molar Refractivity (MR) | Ukuran terkait volume dan polarizabilitas molekul |
| 4 | Hydrogen Bond Acceptor (HBA) | Kemampuan menerima ikatan hidrogen |
| 5 | Rotatable Bond Count | Fleksibilitas struktural |
| 6 | LogP | Lipofilisitas |
| 7 | Hydrogen Bond Donor (HBD) | Kemampuan mendonorkan ikatan hidrogen |
| 8 | Fraction CSp3 | Proporsi karakter alifatik/sp3 pada karbon |
| 9 | Aromatic Heavy Atom Count | Kandungan atom berat dalam sistem aromatik |
| 10 | Topological Polar Surface Area (TPSA) | Karakteristik luas permukaan polar |

Kesepuluh descriptor tersebut dipilih karena memberikan informasi fisikokimia
yang bersifat kontinu dan lebih mudah diinterpretasikan dibandingkan fingerprint.
Set descriptor ini juga konsisten dengan descriptor yang digunakan dalam
rancangan penelitian dan studi EGFR berbasis machine learning yang menjadi
referensi penelitian ini. :chatgpt-content-reference{index="3"}

### 3. Alasan Menggunakan Representasi Hybrid

Fingerprint dan molecular descriptors memberikan informasi yang berbeda.

**ECFP4** berfokus pada pola struktur lokal, sedangkan **molecular descriptors**
menggambarkan karakteristik global seperti ukuran molekul, lipofilisitas,
polaritas, fleksibilitas, dan karakteristik aromatik.

Penggabungan keduanya membentuk representasi:

\[
X = [ECFP4_{2048} \; || \; Descriptors_{10}]
\]

sehingga total terdapat **2058 fitur awal**.

Pendekatan hybrid ini relevan dengan tujuan penelitian karena penelitian tidak
hanya mengevaluasi performa prediksi pada senyawa yang serupa dengan data
pelatihan, tetapi juga menguji **generalisasi terhadap Bemis–Murcko scaffold
yang tidak terdapat pada data pelatihan**. Representasi struktur yang lebih
detail melalui ECFP4 dipadukan dengan karakteristik fisikokimia global agar
model memiliki informasi molecular representation yang lebih lengkap ketika
menghadapi chemical space yang berbeda.

Alshehade et al. menekankan bahwa scaffold-based splitting digunakan untuk
mengukur kemampuan model melakukan generalisasi terhadap chemical series
yang secara struktural baru. Mereka juga menggunakan representasi gabungan
descriptor dan ECFP4 sebagai salah satu konfigurasi QSAR yang dibenchmark.
:chatgpt-content-reference{index="4"}

### 4. Relevansi terhadap Virtual Screening

Representasi ini juga mendukung penggunaan model sebagai model prediksi
potensi yang dapat digunakan untuk **virtual screening**. Pada penelitian
Akinnusi et al., kombinasi ECFP dan physicochemical descriptors digunakan
untuk membangun model prediksi pIC50, kemudian model tersebut digunakan
untuk memprioritaskan senyawa dari DrugBank sebelum dilanjutkan ke molecular
docking. Model ECFP + descriptor tetap memberikan performa pada Murcko
scaffold split, sehingga digunakan sebagai dasar untuk tahap screening.
:chatgpt-content-reference{index="5"}

Dengan demikian, representasi ECFP4 + 10 molecular descriptors digunakan
sebagai representasi utama penelitian ini dan diterapkan secara konsisten pada
seluruh model regresi agar perbandingan model berfokus pada **algoritma dan
kemampuan generalisasi**, bukan pada perbedaan representasi fitur.

In [3]:
for target, df in datasets.items():
    mols = df["canonical_smiles"].apply(Chem.MolFromSmiles)

    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=2048
    )

    fingerprints = [
        generator.GetFingerprint(mol)
        for mol in mols
    ]

    X_ecfp4 = np.array(
        [list(fp) for fp in fingerprints],
        dtype=np.uint8
    )

    print(f"\n{'='*60}")
    print(target)
    print(f"{'='*60}")
    print(f"Compounds   : {len(df):,}")
    print(f"FP shape    : {X_ecfp4.shape}")
    print(f"Active bits : {np.count_nonzero(X_ecfp4.sum(axis=0)):,}")
    print(f"Bit density : {X_ecfp4.mean():.4f}")


EGFR
Compounds   : 10,304
FP shape    : (10304, 2048)
Active bits : 2,048
Bit density : 0.0304

BRAF
Compounds   : 5,353
FP shape    : (5353, 2048)
Active bits : 2,013
Bit density : 0.0308


In [5]:
descriptor_names = [
    "MW",
    "HeavyAtomCount",
    "MolarRefractivity",
    "HBA",
    "RotatableBondCount",
    "LogP",
    "HBD",
    "FractionCSP3",
    "AromaticHeavyAtomCount",
    "TPSA"
]


def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    aromatic_heavy_atoms = sum(
        atom.GetIsAromatic() and atom.GetAtomicNum() > 1
        for atom in mol.GetAtoms()
    )

    return [
        Descriptors.MolWt(mol),
        Descriptors.HeavyAtomCount(mol),
        Descriptors.MolMR(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),
        aromatic_heavy_atoms,
        rdMolDescriptors.CalcTPSA(mol)
    ]


descriptor_data = {}

for target, df in datasets.items():
    values = df["canonical_smiles"].apply(calculate_descriptors)

    desc_df = pd.DataFrame(
        values.tolist(),
        columns=descriptor_names
    )

    descriptor_data[target] = desc_df

    print(f"\n{'='*60}")
    print(target)
    print(f"{'='*60}")
    print(f"Shape   : {desc_df.shape}")
    print(f"Missing : {desc_df.isna().sum().sum()}")

    display(desc_df.describe().T)


EGFR
Shape   : (10304, 10)
Missing : 0


,count,mean,std,min,25%,50%,75%,max
MW,10304.0,482.600248,120.033308,110.1120,403.227500,482.570000,555.495000,1425.8050
HeavyAtomCount,10304.0,34.032512,8.590584,8.0000,28.000000,34.000000,40.000000,103.0000
MolarRefractivity,10304.0,131.710705,32.454332,29.7716,110.193100,131.320700,152.618800,397.0871
HBA,10304.0,6.752426,2.174103,1.0000,5.000000,7.000000,8.000000,19.0000
RotatableBondCount,10304.0,6.918964,3.204919,0.0000,5.000000,7.000000,9.000000,36.0000
LogP,10304.0,4.574709,1.426557,-1.9800,3.669050,4.522600,5.467000,13.0289
HBD,10304.0,2.197108,1.187084,0.0000,1.000000,2.000000,3.000000,17.0000
FractionCSP3,10304.0,0.236541,0.126938,0.0000,0.142857,0.241379,0.321429,1.0000
AromaticHeavyAtomCount,10304.0,19.052019,4.664084,0.0000,16.000000,18.000000,22.000000,52.0000
TPSA,10304.0,96.317341,30.219542,6.4800,77.530000,96.450000,111.720000,530.8700



BRAF
Shape   : (5353, 10)
Missing : 0


,count,mean,std,min,25%,50%,75%,max
MW,5353.0,488.915019,78.980980,247.2790,442.9500,487.4820,531.519000,1412.367000
HeavyAtomCount,5353.0,34.991780,5.540745,17.0000,32.0000,35.0000,38.000000,93.000000
MolarRefractivity,5353.0,130.101097,20.625246,64.9729,117.0497,128.9726,141.536200,335.519400
HBA,5353.0,6.521950,1.838473,2.0000,5.0000,6.0000,8.000000,19.000000
RotatableBondCount,5353.0,5.956660,2.083458,0.0000,5.0000,6.0000,7.000000,34.000000
LogP,5353.0,4.502146,1.137962,-0.5535,3.7789,4.4989,5.213700,11.605560
HBD,5353.0,2.194657,0.942793,0.0000,1.0000,2.0000,3.000000,11.000000
FractionCSP3,5353.0,0.244494,0.122901,0.0000,0.1500,0.2500,0.346154,0.714286
AromaticHeavyAtomCount,5353.0,20.334766,3.627099,6.0000,18.0000,21.0000,22.000000,42.000000
TPSA,5353.0,103.449705,22.937502,29.8500,89.0300,102.3600,117.360000,295.450000


In [6]:
for target, desc_df in descriptor_data.items():
    print(f"\n{'='*60}")
    print(target)
    print(f"{'='*60}")

    constant = desc_df.columns[desc_df.nunique() <= 1]

    corr = desc_df.corr().abs()
    high_corr = [
        (corr.columns[i], corr.columns[j], corr.iloc[i, j])
        for i in range(len(corr.columns))
        for j in range(i + 1, len(corr.columns))
        if corr.iloc[i, j] >= 0.90
    ]

    print(f"Constant descriptors : {len(constant)}")

    if len(constant) > 0:
        print("Constant:", list(constant))

    print(f"High correlation pairs (|r| >= 0.90): {len(high_corr)}")

    if high_corr:
        display(
            pd.DataFrame(
                high_corr,
                columns=["Feature 1", "Feature 2", "|Correlation|"]
            ).sort_values("|Correlation|", ascending=False)
        )


EGFR
Constant descriptors : 0
High correlation pairs (|r| >= 0.90): 3


,Feature 1,Feature 2,|Correlation|
2,HeavyAtomCount,MolarRefractivity,0.983393
0,MW,HeavyAtomCount,0.972579
1,MW,MolarRefractivity,0.968420



BRAF
Constant descriptors : 0
High correlation pairs (|r| >= 0.90): 3


,Feature 1,Feature 2,|Correlation|
0,MW,HeavyAtomCount,0.966224
2,HeavyAtomCount,MolarRefractivity,0.952778
1,MW,MolarRefractivity,0.915827


## Feature Assembly

In [7]:
feature_data = {}

generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

for target, df in datasets.items():
    fingerprints = [
        generator.GetFingerprint(Chem.MolFromSmiles(smiles))
        for smiles in df["canonical_smiles"]
    ]

    ecfp4 = np.array(
        [list(fp) for fp in fingerprints],
        dtype=np.uint8
    )

    descriptors = descriptor_data[target].to_numpy(dtype=np.float32)

    X = np.hstack([ecfp4, descriptors])
    y = df["pIC50"].to_numpy(dtype=np.float32)

    feature_data[target] = {
        "X": X,
        "y": y
    }

    print(f"\n{'='*60}")
    print(target)
    print(f"{'='*60}")
    print(f"Compounds : {len(df):,}")
    print(f"ECFP4     : {ecfp4.shape}")
    print(f"Descriptors: {descriptors.shape}")
    print(f"Final X   : {X.shape}")
    print(f"Target y  : {y.shape}")
    print(f"Missing X : {np.isnan(X).sum():,}")


EGFR
Compounds : 10,304
ECFP4     : (10304, 2048)
Descriptors: (10304, 10)
Final X   : (10304, 2058)
Target y  : (10304,)
Missing X : 0

BRAF
Compounds : 5,353
ECFP4     : (5353, 2048)
Descriptors: (5353, 10)
Final X   : (5353, 2058)
Target y  : (5353,)
Missing X : 0


In [8]:
FEATURE_DIR = Path("../dataset/dataset_features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

feature_columns = (
    [f"ECFP4_{i}" for i in range(2048)] +
    descriptor_names
)

for target, data in feature_data.items():
    df_meta = datasets[target][
        ["canonical_smiles", "molecule_chembl_id", "pIC50"]
    ].copy()

    X = data["X"]

    X_df = pd.DataFrame(
        X,
        columns=feature_columns,
        index=df_meta.index
    )

    feature_df = pd.concat(
        [df_meta, X_df],
        axis=1
    )

    path = FEATURE_DIR / f"{target}_features_ECFP4_DESC.csv"
    feature_df.to_csv(path, index=False)

    print(f"\n{'='*60}")
    print(target)
    print(f"{'='*60}")
    print(f"Shape : {feature_df.shape}")
    print(f"Saved : {path}")


EGFR
Shape : (10304, 2061)
Saved : ..\dataset\dataset_features\EGFR_features_ECFP4_DESC.csv

BRAF
Shape : (5353, 2061)
Saved : ..\dataset\dataset_features\BRAF_features_ECFP4_DESC.csv
